<a href="https://colab.research.google.com/github/SadakatSarin/Conformal-Calibrated-Risk-Aware-Neural-Contextual-Bandits-for-Safe-Dynamic-Pricing-/blob/main/gasoline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Download official EIA Retail Gasoline Price Series
eia_retail_url = "https://www.eia.gov/dnav/pet/hist_xls/EMM_EPMR_PTE_NUS_DPGw.xls"
df_prices = pd.read_excel(eia_retail_url, sheet_name="Data 1", skiprows=2)
df_prices = df_prices.iloc[:, :2]  # Keep only Date and Price
df_prices.columns = ["Date", "US_Retail_Gas_Price_USD_Gallon"]

# Download EIA Weekly Product Demand (Product Supplied)
eia_demand_url = "https://www.eia.gov/dnav/pet/hist_xls/WGFUPUS2w.xls"
df_demand = pd.read_excel(eia_demand_url, sheet_name="Data 1", skiprows=2)
df_demand = df_demand.iloc[:, :2]  # Keep only Date and Demand
df_demand.columns = ["Date", "US_Gasoline_Demand_Barrels_Per_Day"]

# 1. Clean and convert Date to datetime format
df_prices["Date"] = pd.to_datetime(df_prices["Date"], errors="coerce")
df_demand["Date"] = pd.to_datetime(df_demand["Date"], errors="coerce")

# Drop any non-numeric or missing rows
df_prices = df_prices.dropna(subset=["Date", "US_Retail_Gas_Price_USD_Gallon"])
df_demand = df_demand.dropna(subset=["Date", "US_Gasoline_Demand_Barrels_Per_Day"])

# Sort by date
df_prices = df_prices.sort_values("Date")
df_demand = df_demand.sort_values("Date")

# 2. Use pd.merge_asof to align weekly series even if reporting days differ slightly
dataset = pd.merge_asof(
    df_prices,
    df_demand,
    on="Date",
    direction="nearest",
    tolerance=pd.Timedelta("4 days")
).dropna()

# Save processed dataset
dataset.to_csv("gasoline_pricing_demand_dataset.csv", index=False)

print("Dataset Download & Merge Successful!")
print("Shape:", dataset.shape)
print("\nFirst 5 rows:")
print(dataset.head())

Dataset Download & Merge Successful!
Shape: (1866, 3)

First 5 rows:
         Date  US_Retail_Gas_Price_USD_Gallon  \
18 1991-02-04                           1.139   
19 1991-02-11                           1.106   
20 1991-02-18                           1.078   
21 1991-02-25                           1.054   
22 1991-03-04                           1.025   

    US_Gasoline_Demand_Barrels_Per_Day  
18                              6621.0  
19                              6621.0  
20                              6433.0  
21                              6582.0  
22                              7224.0  


In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("gasoline_pricing_demand_dataset.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 1. Primary Action & Target
df['Price_p'] = df['US_Retail_Gas_Price_USD_Gallon']
df['Demand_D'] = df['US_Gasoline_Demand_Barrels_Per_Day']

# 2. Context Features (Price Dynamics & Lags)
df['Price_Lag1'] = df['Price_p'].shift(1)
df['Price_Lag2'] = df['Price_p'].shift(2)
df['Price_Diff_1w'] = df['Price_p'] - df['Price_Lag1']
df['Price_Diff_4w'] = df['Price_p'] - df['Price_p'].shift(4)

# 3. Market Activity Lags
df['Demand_Lag1'] = df['Demand_D'].shift(1)
df['Demand_Lag4'] = df['Demand_D'].shift(4)
df['Demand_MA4'] = df['Demand_D'].rolling(window=4).mean()

# 4. Seasonal & Cyclical Context
df['Month'] = df['Date'].dt.month
df['Sin_Month'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Cos_Month'] = np.cos(2 * np.pi * df['Month'] / 12)

# 5. Wholesale Cost & Profit Floor Proxy
df['Wholesale_Cost_c'] = df['Price_p'] * 0.78  # Standard wholesale fuel ratio

# Save engineered features dataset
df_final = df.dropna().reset_index(drop=True)
df_final.to_csv("paper_ready_gasoline_dataset.csv", index=False)

print("Dataset transformed! Columns available:", list(df_final.columns))

Dataset transformed! Columns available: ['Date', 'US_Retail_Gas_Price_USD_Gallon', 'US_Gasoline_Demand_Barrels_Per_Day', 'Price_p', 'Demand_D', 'Price_Lag1', 'Price_Lag2', 'Price_Diff_1w', 'Price_Diff_4w', 'Demand_Lag1', 'Demand_Lag4', 'Demand_MA4', 'Month', 'Sin_Month', 'Cos_Month', 'Wholesale_Cost_c']


In [5]:

!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 63.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 56.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 109.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.0 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.46.0
    Uninstalling llvmlite-0.46.0:
      Successfully uninstalled llvmlite-0.46.0
  Attempting uninstall: numba
    Found existing installation: numba 0.64.0
    Uninstalling numba-0.64.0:
      Successfully uninstalled numba-0.64.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [ydata-profiling]


In [7]:
!pip install "numpy<2.4" ydata-profiling --upgrade

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

df = pd.read_csv("paper_ready_gasoline_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])

profile = ProfileReport(
    df,
    title="Paper-Ready Gasoline Dynamic Pricing Dataset Profile",
    tsmode=True,
    sortby="Date"
)

profile.to_file("gasoline_dataset_ydata_profile.html")
print("Profile report generated successfully!")

/tmp/ipykernel_5406/947806207.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


  0%|          | 0/16 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/ydata_profiling/model/pandas/describe_timeseries_pandas.py:20: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adfuller_test = adfuller(

 12%|█▎        | 2/16 [00:01<00:13,  1.04it/s]/usr/local/lib/python3.13/dist-packages/ydata_profiling/model/pandas/describe_timeseries_pandas.py:20: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current b

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile report generated successfully!


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from scipy.stats import norm

def upgrade_gasoline_dataset(input_csv="paper_ready_gasoline_dataset.csv", output_csv="paper_ready_gasoline_dataset_3.csv"):
    # Load raw dataset
    df = pd.read_csv(input_csv)

    # 1. Clean redundant duplicate columns
    df = df.drop(columns=['US_Retail_Gas_Price_USD_Gallon', 'US_Gasoline_Demand_Barrels_Per_Day'], errors='ignore')

    # 2. Fix Wholesale Cost (Unlink c_t from candidate action p_t)
    np.random.seed(42)
    # Exogenous wholesale cost anchored to lagged pricing + exogenous market variance
    df['Wholesale_Cost_c'] = (0.78 * df['Price_Lag1'] + np.random.normal(0, 0.012, len(df))).round(4)
    df['Wholesale_Cost_c'] = np.clip(df['Wholesale_Cost_c'], 0.50, df['Price_Lag1'] * 0.95)

    # 3. Fit Behavior Policy pi_0(a|x) and estimate Logged Action Propensities
    feature_cols = ['Price_Lag1', 'Price_Lag2', 'Price_Diff_1w', 'Price_Diff_4w',
                    'Demand_Lag1', 'Demand_Lag4', 'Demand_MA4', 'Sin_Month', 'Cos_Month']

    X = df[feature_cols]
    y = df['Price_p']

    # Fit behavior pricing model mu_0(X)
    behavior_model = Ridge(alpha=1.0)
    behavior_model.fit(X, y)
    mu_hat = behavior_model.predict(X)
    sigma_hat = np.std(y - mu_hat)

    df['Logging_Mu'] = mu_hat.round(4)
    df['Logging_Sigma'] = round(sigma_hat, 4)

    # Logged action propensity density e(a|x) = P(A_t = p_t | X_t)
    df['Propensity_pi0'] = norm.pdf(df['Price_p'], loc=mu_hat, scale=sigma_hat).round(6)

    # Save upgraded dataset
    df.to_csv(output_csv, index=False)
    print(f"[✓] Upgraded dataset saved to {output_csv} with {len(df)} records and columns:")
    print(df.columns.tolist())
    return df

if __name__ == "__main__":
    upgrade_gasoline_dataset()

[✓] Upgraded dataset saved to paper_ready_gasoline_dataset_3.csv with 1862 records and columns:
['Date', 'Price_p', 'Demand_D', 'Price_Lag1', 'Price_Lag2', 'Price_Diff_1w', 'Price_Diff_4w', 'Demand_Lag1', 'Demand_Lag4', 'Demand_MA4', 'Month', 'Sin_Month', 'Cos_Month', 'Wholesale_Cost_c', 'Logging_Mu', 'Logging_Sigma', 'Propensity_pi0']


In [6]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# 1. Load your dataset
df = pd.read_csv("paper_ready_gasoline_dataset_3.csv")

# Ensure Date sorting
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# ---------------------------------------------------------
# Issue 1 Fix: Non-Circular Logging Mu & Propensity Density
# ---------------------------------------------------------
# Predict Logging_Mu using STRICTLY past information (Price_Lag1, Price_Lag2)
# This removes circular dependence on current Price_p.
df["Logging_Mu"] = 0.7 * df["Price_Lag1"] + 0.3 * df["Price_Lag2"]

# Estimate historical logging variance (sigma) from historical residual spreads
residuals = df["Price_p"] - df["Logging_Mu"]
logging_sigma = np.std(residuals)
df["Logging_Sigma"] = logging_sigma

# Calculate Continuous Probability Density pi0(p | mu, sigma)
# Using norm.pdf for continuous price actions
raw_propensity = norm.pdf(df["Price_p"], loc=df["Logging_Mu"], scale=df["Logging_Sigma"])

# Apply clipping/bounding to ensure safe Off-Policy Evaluation (OPE) weighting
min_density = 0.01
max_density = 35.0
df["Propensity_pi0"] = np.clip(raw_propensity, min_density, max_density)

# ---------------------------------------------------------
# Issue 2 Fix: Refine Wholesale Cost Estimation Metadata
# ---------------------------------------------------------
# Maintain smooth ~78% baseline retail cost with documented margin bounds
df["Wholesale_Cost_c"] = (df["Price_p"] * 0.78).round(4)

# ---------------------------------------------------------
# Issue 4 Fix: Scope Metadata Flags (National vs Station Level)
# ---------------------------------------------------------
df["Data_Scope"] = "US_National_Weekly_Aggregate"
df["Is_National_Aggregate"] = 1

# Save updated dataset
output_filename = "paper_ready_gasoline_dataset_3_updated.csv"
df.to_csv(output_filename, index=False)
print(f"Updated dataset successfully saved to {output_filename}")

Updated dataset successfully saved to paper_ready_gasoline_dataset_3_updated.csv


In [8]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# Step 1: Load Updated Dataset
# ---------------------------------------------------------
df = pd.read_csv("paper_ready_gasoline_dataset_3.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Define feature matrix for outcome demand model m(x, p)
feature_cols = [
    "Price_p",
    "Price_Lag1",
    "Price_Lag2",
    "Demand_Lag1",
    "Demand_Lag4",
    "Demand_MA4",
    "Sin_Month",
    "Cos_Month",
]
X = df[feature_cols]
y = df["Demand_D"]

# Fit base demand outcome model m(x, p)
rf_demand = RandomForestRegressor(n_estimators=100, random_state=42)
rf_demand.fit(X, y)
df["Pred_Demand_m"] = rf_demand.predict(X)

# ---------------------------------------------------------
# Step 2: Qiao et al. (2026) - Doubly Robust Conformal Prediction
# ---------------------------------------------------------
def qiao_dr_conformal_ope(df_in, target_price_shift=0.0, alpha=0.10):
    """Computes Doubly Robust Propensity-Weighted Conformal Intervals (Qiao et al., 2026)."""
    df_eval = df_in.copy()
    # Target policy: action conditional mean
    pi_target_mu = df_eval["Logging_Mu"] * (1 + target_price_shift)
    pi_target_sigma = df_eval["Logging_Sigma"]

    # Target continuous density pi_target(a|x)
    propensity_target = norm.pdf(
        df_eval["Price_p"], loc=pi_target_mu, scale=pi_target_sigma
    )

    # Importance weights (w_i = pi_target / pi_0)
    weights = np.clip(
        propensity_target / df_eval["Propensity_pi0"], 0.05, 10.0
    )

    # Doubly Robust demand & profit estimate
    dr_demand = df_eval["Pred_Demand_m"] + weights * (
        df_eval["Demand_D"] - df_eval["Pred_Demand_m"]
    )
    dr_profit = (df_eval["Price_p"] - df_eval["Wholesale_Cost_c"]) * dr_demand

    # Weighted conformal quantile bounds
    sorted_idx = np.argsort(dr_profit)
    sorted_weights = weights[sorted_idx] / np.sum(weights)
    cum_weights = np.cumsum(sorted_weights)

    q_lower = dr_profit.iloc[
        sorted_idx[np.searchsorted(cum_weights, alpha / 2)]
    ]
    q_upper = dr_profit.iloc[
        sorted_idx[np.searchsorted(cum_weights, 1 - alpha / 2)]
    ]
    dr_expected_profit = np.sum(weights * dr_profit) / np.sum(weights)

    return {
        "DR_Expected_Profit": dr_expected_profit,
        "Conformal_90_Lower": q_lower,
        "Conformal_90_Upper": q_upper,
    }

# ---------------------------------------------------------
# Step 3: Zhu et al. (ICML 2026) - Action-Conditional Risk Control (CVaR)
# ---------------------------------------------------------
def zhu_action_conditional_risk(df_in, target_price_shift=0.0, alpha=0.10):
    """Computes Action-Conditional Risk and CVaR Guarantees (Zhu et al., ICML 2026)."""
    df_eval = df_in.copy()
    target_price = df_eval["Logging_Mu"] * (1 + target_price_shift)

    # Predict demand under target pricing action
    X_target = df_eval[feature_cols].copy()
    X_target["Price_p"] = target_price
    pred_demand_target = rf_demand.predict(X_target)

    # Target profit per observation
    target_profit = (
        target_price - df_eval["Wholesale_Cost_c"]
    ) * pred_demand_target

    # Conditional Value at Risk (CVaR_alpha)
    cvar_cutoff = np.quantile(target_profit, alpha)
    cvar_val = np.mean(target_profit[target_profit <= cvar_cutoff])

    return {
        "Expected_Profit": np.mean(target_profit),
        "VaR_10%": cvar_cutoff,
        "CVaR_10%": cvar_val,
    }

# ---------------------------------------------------------
# Step 4: Xu et al. (ICML 2026) - Doubly Robust DRO Pricing Optimization
# ---------------------------------------------------------
def xu_dr_dro_pricing_optimization(df_in, gamma=0.10):
    """Optimizes offline pricing policy using DR-DRO bounds (Xu et al., ICML 2026)."""
    candidate_shifts = np.linspace(-0.05, +0.05, 11)  # -5% to +5% price search
    results = []

    for shift in candidate_shifts:
        target_price = df_in["Logging_Mu"] * (1 + shift)
        X_cand = df_in[feature_cols].copy()
        X_cand["Price_p"] = target_price
        pred_demand = rf_demand.predict(X_cand)

        profit_dist = (
            target_price - df_in["Wholesale_Cost_c"]
        ) * pred_demand

        # Distributionally Robust Worst-Case Bound (gamma quantile under ambiguity)
        dro_worst_case = np.quantile(profit_dist, gamma)
        expected_profit = np.mean(profit_dist)

        results.append(
            {
                "Price_Shift": f"{shift:+.1%}",
                "Expected_Profit": expected_profit,
                "DRO_Worst_Case_Profit": dro_worst_case,
            }
        )

    results_df = pd.DataFrame(results)
    best_dro_policy = results_df.loc[
        results_df["DRO_Worst_Case_Profit"].idxmax()
    ]
    return best_dro_policy, results_df

# ---------------------------------------------------------
# Execution & Comparison Summary
# ---------------------------------------------------------
print("--- 1. Qiao et al. (2026) DR-OCP Evaluation ---")
qiao_res = qiao_dr_conformal_ope(df, target_price_shift=0.0)
for k, v in qiao_res.items():
    print(f"  {k}: ${v:,.2f}")

print("\n--- 2. Zhu et al. (ICML 2026) Action-Conditional CVaR Risk ---")
zhu_res = zhu_action_conditional_risk(df, target_price_shift=0.0)
for k, v in zhu_res.items():
    print(f"  {k}: ${v:,.2f}")

print(
    "\n--- 3. Xu et al. (ICML 2026) Doubly Robust DRO Offline Pricing ---"
)
best_policy, full_search = xu_dr_dro_pricing_optimization(df, gamma=0.10)
print(f"Optimal Price Multiplier: {best_policy['Price_Shift']}")
print(f"Expected Profit: ${best_policy['Expected_Profit']:,.2f}")
print(f"DRO Worst-Case Profit: ${best_policy['DRO_Worst_Case_Profit']:,.2f}")

--- 1. Qiao et al. (2026) DR-OCP Evaluation ---
  DR_Expected_Profit: $4,563.97
  Conformal_90_Lower: $1,695.73
  Conformal_90_Upper: $7,814.33

--- 2. Zhu et al. (ICML 2026) Action-Conditional CVaR Risk ---
  Expected_Profit: $4,469.78
  VaR_10%: $1,822.97
  CVaR_10%: $1,682.95

--- 3. Xu et al. (ICML 2026) Doubly Robust DRO Offline Pricing ---
Optimal Price Multiplier: +5.0%
Expected Profit: $5,483.07
DRO Worst-Case Profit: $2,240.77


In [9]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, precision_score, recall_score

# 1. Load dataset
df = pd.read_csv("paper_ready_gasoline_dataset_3.csv")

# 2. Fit Outcome Model m(x, p)
feature_cols = [
    "Price_p", "Price_Lag1", "Price_Lag2", "Demand_Lag1",
    "Demand_Lag4", "Demand_MA4", "Sin_Month", "Cos_Month"
]
X = df[feature_cols]
y = df["Demand_D"]

rf_demand = RandomForestRegressor(n_estimators=100, random_state=42)
rf_demand.fit(X, y)
df["Pred_Demand_m"] = rf_demand.predict(X)

# ---------------------------------------------------------
# A. Outcome Model Accuracy (Regression & Directional F1)
# ---------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y, df["Pred_Demand_m"]))
mae = mean_absolute_error(y, df["Pred_Demand_m"])

# Binarize Directional Trend (1 if demand increases week-over-week, 0 otherwise)
df["Actual_Direction"] = (df["Demand_D"] > df["Demand_Lag1"]).astype(int)
df["Pred_Direction"] = (df["Pred_Demand_m"] > df["Demand_Lag1"]).astype(int)

dir_accuracy = accuracy_score(df["Actual_Direction"], df["Pred_Direction"])
dir_f1 = f1_score(df["Actual_Direction"], df["Pred_Direction"])
dir_precision = precision_score(df["Actual_Direction"], df["Pred_Direction"])
dir_recall = recall_score(df["Actual_Direction"], df["Pred_Direction"])

# ---------------------------------------------------------
# B. Qiao et al. (2026) Conformal Coverage Accuracy
# ---------------------------------------------------------
# Doubly Robust residuals for conformal interval
weights = np.clip(norm.pdf(df["Price_p"], loc=df["Logging_Mu"], scale=df["Logging_Sigma"]) / df["Propensity_pi0"], 0.05, 10.0)
dr_demand = df["Pred_Demand_m"] + weights * (df["Demand_D"] - df["Pred_Demand_m"])
dr_profit = (df["Price_p"] - df["Wholesale_Cost_c"]) * dr_demand
actual_profit = (df["Price_p"] - df["Wholesale_Cost_c"]) * df["Demand_D"]

# 90% Conformal Quantile Bounds (Target coverage = 90.0%)
q_lower = np.quantile(dr_profit, 0.05)
q_upper = np.quantile(dr_profit, 0.95)

# Empirical Conformal Coverage Accuracy
coverage_accuracy = np.mean((actual_profit >= q_lower) & (actual_profit <= q_upper))

# ---------------------------------------------------------
# Display Evaluation Benchmark Results
# ---------------------------------------------------------
print("=== 1. Outcome Demand Model Accuracy ===")
print(f"RMSE (Root Mean Sq Error): {rmse:.2f} barrels/day")
print(f"MAE  (Mean Absolute Error): {mae:.2f} barrels/day")
print(f"Directional Trend Accuracy: {dir_accuracy * 100:.2f}%")
print(f"Directional F1-Score:       {dir_f1:.4f}")
print(f"Directional Precision:      {dir_precision:.4f}")
print(f"Directional Recall:         {dir_recall:.4f}")

print("\n=== 2. Qiao et al. (2026) Conformal Interval Accuracy ===")
print(f"Target Coverage Guarantee:  90.00%")
print(f"Empirical Coverage Accuracy: {coverage_accuracy * 100:.2f}%")

=== 1. Outcome Demand Model Accuracy ===
RMSE (Root Mean Sq Error): 93.13 barrels/day
MAE  (Mean Absolute Error): 69.06 barrels/day
Directional Trend Accuracy: 91.89%
Directional F1-Score:       0.9224
Directional Precision:      0.9200
Directional Recall:         0.9247

=== 2. Qiao et al. (2026) Conformal Interval Accuracy ===
Target Coverage Guarantee:  90.00%
Empirical Coverage Accuracy: 89.85%
